In [971]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import shapiro, mannwhitneyu, ttest_ind, levene
import warnings
warnings.filterwarnings('ignore')

# For VIF calculation (manual implementation without statsmodels)
from sklearn.linear_model import LinearRegression


In [972]:
# SECTION 1: LOAD ALL CLEANED DATASETS
prio = pd.read_csv('/Users/Marcy_Student/Desktop/Food Insecurity Analysis/datasets/cleaned_for_eda&modeling/cleaned_neighborhood_prioritization.csv')
efap = pd.read_csv('/Users/Marcy_Student/Desktop/Marcy_Projects/CID_Food_Access/data/clean/efap_cleaned.csv')
efap_nta_mapping = pd.read_csv('/Users/Marcy_Student/Desktop/Marcy_Projects/CID_Food_Access/data/clean/efap_nta_mapping.csv')
dim_map = pd.read_csv('/Users/Marcy_Student/Desktop/Marcy_Projects/CID_Food_Access/data/clean/dim_map.csv')
shelter_census = pd.read_csv('/Users/Marcy_Student/Desktop/Marcy_Projects/CID_Food_Access/data/clean/shelter_census_clean.csv')

In [973]:
print(f"\n1. Neighborhood Prioritization: {prio.shape[0]} rows, {prio.shape[1]} columns")
print(f"2. EFAP Programs: {efap.shape[0]} rows, {efap.shape[1]} columns")
print(f"3. EFAP-NTA Mapping: {efap_nta_mapping.shape[0]} rows, {efap_nta_mapping.shape[1]} columns")
print(f"4. Dimension Map (Geography): {dim_map.shape[0]} rows, {dim_map.shape[1]} columns")
print(f"5. Shelter Census: {shelter_census.shape[0]} rows, {shelter_census.shape[1]} columns")


1. Neighborhood Prioritization: 197 rows, 11 columns
2. EFAP Programs: 561 rows, 7 columns
3. EFAP-NTA Mapping: 553 rows, 4 columns
4. Dimension Map (Geography): 262 rows, 7 columns
5. Shelter Census: 5204 rows, 6 columns


In [974]:
# Quick peek at each dataset
print("\n--- Dataset Previews ---")
print("\nPrioritization columns:", list(prio.columns))
print("EFAP columns:", list(efap.columns))
print("EFAP-NTA Mapping columns:", list(efap_nta_mapping.columns))
print("Dim Map columns:", list(dim_map.columns))
print("Shelter Census columns:", list(shelter_census.columns))


--- Dataset Previews ---

Prioritization columns: ['nta_id', 'nta_name', 'borough', 'food_insecure_percentage', 'food_insecure_percentage_rank', 'unemployment_rate', 'unemployment_rate_rank', 'vulnerable_population_percentage', 'vulnerable_population_percentage_rank', 'supply_gap', 'weighted_score']
EFAP columns: ['efap_id', 'program_name', 'access_type', 'has_pantry_access', 'has_kitchen_access', 'weekday_available', 'weekend_available']
EFAP-NTA Mapping columns: ['efap_id', 'nta_id', 'lat', 'lon']
Dim Map columns: ['nta_id', 'nta_name', 'cdta_id', 'cdta_name', 'boro_code', 'boro_name', 'the_geom_wkt']
Shelter Census columns: ['report_date', 'borough', 'community_districts', 'family_with_children_commercial_hotel', 'family_with_children_shelter', 'family_cluster']


In [975]:
# Merge EFAP with EFAP-NTA Mapping to get NTA for each site
efap_with_nta = efap.merge(efap_nta_mapping[['efap_id', 'nta_id']], on='efap_id', how='left')

In [976]:
efap_with_nta.head()


,efap_id,program_name,access_type,has_pantry_access,has_kitchen_access,weekday_available,weekend_available,nta_id
0,80604,HOLY APOSTLES SOUP KITCHEN,Kitchen,0,1,1,0,MN0401
1,85547,HOLY APOSTLES SOUP KITCHEN PANTRY,Pantry,1,0,1,0,MN0401
2,80757,ST. JOHN'S BREAD OF LIFE,Pantry,1,0,1,0,MN0501
3,85701,ARTISTS ATHLETES ACTIVISTS INCORPORATED,Pantry,1,0,1,0,MN0302
4,80546,DEWITT REFORMED CHURCH,Pantry,1,0,0,1,MN0302


In [977]:
print(f"EFAP sites with NTA mapping: {efap_with_nta['nta_id'].notna().sum()} of {len(efap_with_nta)}")
print(f"EFAP sites without NTA (will be excluded): {efap_with_nta['nta_id'].isna().sum()}")

EFAP sites with NTA mapping: 553 of 561
EFAP sites without NTA (will be excluded): 8


In [978]:
# Drop sites without NTA mapping
efap_with_nta = efap_with_nta.dropna(subset=['nta_id'])
print(f"EFAP sites after dropping unmapped: {len(efap_with_nta)}")

EFAP sites after dropping unmapped: 553


In [979]:
print("\n--- Step 2.2: Aggregate EFAP Sites to NTA Level ---")

nta_efap_agg = efap_with_nta.groupby('nta_id').agg(
    total_sites=('efap_id', 'count'),
    pantry_sites=('has_pantry_access', 'sum'),
    kitchen_sites=('has_kitchen_access', 'sum'),
    weekday_sites=('weekday_available', 'sum'),
    weekend_sites=('weekend_available', 'sum')
).reset_index()
print(f"Aggregated EFAP data at NTA level: {nta_efap_agg.shape[0]} NTAs")
nta_efap_agg


--- Step 2.2: Aggregate EFAP Sites to NTA Level ---
Aggregated EFAP data at NTA level: 152 NTAs


,nta_id,total_sites,pantry_sites,kitchen_sites,weekday_sites,weekend_sites
0,BK0101,1,1,0,1,0
1,BK0102,1,1,0,1,0
2,BK0103,1,1,0,1,0
3,BK0104,1,1,0,1,0
4,BK0202,3,2,1,2,1
...,...,...,...,...,...,...
147,SI0201,1,1,0,1,0
148,SI0202,1,1,0,1,0
149,SI0203,1,1,1,1,0
150,SI0302,1,1,0,1,0


In [980]:
# Parse date and filter to 2023-2024
shelter_census['report_date'] = pd.to_datetime(shelter_census['report_date'])
shelter_census['year'] = shelter_census['report_date'].dt.year
shelter_census['month'] = shelter_census['report_date'].dt.month
shelter_census['Year-Month'] = shelter_census['report_date'].dt.to_period('M')

# Filter to 2023-2024
shelter_2023_2024 = shelter_census[shelter_census['year'].isin([2023, 2024])].copy()
print(f"Shelter records for 2023-2024: {len(shelter_2023_2024)}")

Shelter records for 2023-2024: 1436


In [981]:
shelter_2023_2024.isna().sum()

report_date                                 0
borough                                     0
community_districts                        20
family_with_children_commercial_hotel     563
family_with_children_shelter              387
family_cluster                           1436
year                                        0
month                                       0
Year-Month                                  0
dtype: int64

In [982]:
# Fill NaN with 0 for shelter counts (assumption: NaN = no recorded families)
shelter_cols = ['family_with_children_commercial_hotel', 'family_with_children_shelter', 'family_cluster']
for col in shelter_cols:
    shelter_2023_2024[col] = shelter_2023_2024[col].fillna(0).astype(int)



In [983]:
shelter_2023_2024

,report_date,borough,community_districts,family_with_children_commercial_hotel,family_with_children_shelter,family_cluster,year,month,Year-Month
759,2024-12-31,Brooklyn,15.0,425,182,0,2024,12,2024-12
760,2024-12-31,Brooklyn,16.0,808,2376,0,2024,12,2024-12
761,2024-12-31,Brooklyn,17.0,319,1600,0,2024,12,2024-12
762,2024-12-31,Brooklyn,18.0,329,819,0,2024,12,2024-12
763,2024-12-31,Manhattan,1.0,527,0,0,2024,12,2024-12
...,...,...,...,...,...,...,...,...,...
2190,2023-01-31,Brooklyn,15.0,0,204,0,2023,1,2023-01
2191,2023-01-31,Brooklyn,14.0,0,88,0,2023,1,2023-01
2192,2023-01-31,Brooklyn,13.0,0,463,0,2023,1,2023-01
2193,2023-01-31,Brooklyn,12.0,0,144,0,2023,1,2023-01


In [984]:
# Aggregate to borough + CD level (sum over 2023-2024)
shelter_cd_agg = shelter_2023_2024.groupby(['borough', 'community_districts']).agg(
    total_families_with_children_shelter=('family_with_children_shelter', 'sum'),
).reset_index()

In [985]:
shelter_cd_agg.head(5)

,borough,community_districts,total_families_with_children_shelter
0,Bronx,1.0,28947
1,Bronx,2.0,20047
2,Bronx,3.0,30888
3,Bronx,4.0,52319
4,Bronx,5.0,38815


In [986]:
# Create high shelter concentration flag (top 25%)
shelter_cd_agg['high_shelter_flag'] = (
    shelter_cd_agg['total_families_with_children_shelter'] >= shelter_cd_agg['total_families_with_children_shelter'].quantile(0.75)
).astype(int)

print(f"Community Districts with shelter data: {len(shelter_cd_agg)}")
print(f"High shelter concentration CDs: {shelter_cd_agg['high_shelter_flag'].sum()}")

Community Districts with shelter data: 59
High shelter concentration CDs: 15


In [987]:
# Extract CD number from cdta_id (e.g., 'BK14' -> 14)
dim_map['cd_number'] = dim_map['cdta_id'].str.extract(r'(\d+)').astype(float)

# Map borough codes to full names
boro_map = {'BK': 'Brooklyn', 'BX': 'Bronx', 'MN': 'Manhattan', 'QN': 'Queens', 'SI': 'Staten Island'}
dim_map['borough_from_cdta'] = dim_map['cdta_id'].str[:2].map(boro_map)

print(f"Unique NTAs in dim_map: {dim_map['nta_id'].nunique()}")
print(f"Unique CDTAs in dim_map: {dim_map['cdta_id'].nunique()}")

# Create NTA to CDTA lookup
nta_to_cdta = dim_map[['nta_id', 'cdta_id', 'cd_number', 'boro_name']].drop_duplicates()
print(f"\nNTA to CDTA mapping sample:")
print(nta_to_cdta.head(10))

Unique NTAs in dim_map: 262
Unique CDTAs in dim_map: 71

NTA to CDTA mapping sample:
   nta_id cdta_id  cd_number boro_name
0  BK0101    BK01        1.0  Brooklyn
1  BK0102    BK01        1.0  Brooklyn
2  BK0103    BK01        1.0  Brooklyn
3  BK0104    BK01        1.0  Brooklyn
4  BK0201    BK02        2.0  Brooklyn
5  BK0202    BK02        2.0  Brooklyn
6  BK0203    BK02        2.0  Brooklyn
7  BK0204    BK02        2.0  Brooklyn
8  BK0261    BK02        2.0  Brooklyn
9  BK0301    BK03        3.0  Brooklyn


In [988]:
# Merge shelter_cd_agg with nta_to_cdta
# Match on borough and community_districts (CD number)
shelter_cd_agg['community_districts'] = shelter_cd_agg['community_districts'].astype(float)

nta_shelter = nta_to_cdta.merge(
    shelter_cd_agg,
    left_on=['boro_name', 'cd_number'],
    right_on=['borough', 'community_districts'],
    how='left'
)

In [989]:
nta_efap_agg

,nta_id,total_sites,pantry_sites,kitchen_sites,weekday_sites,weekend_sites
0,BK0101,1,1,0,1,0
1,BK0102,1,1,0,1,0
2,BK0103,1,1,0,1,0
3,BK0104,1,1,0,1,0
4,BK0202,3,2,1,2,1
...,...,...,...,...,...,...
147,SI0201,1,1,0,1,0
148,SI0202,1,1,0,1,0
149,SI0203,1,1,1,1,0
150,SI0302,1,1,0,1,0


In [990]:
nta_to_cdta

,nta_id,cdta_id,cd_number,boro_name
0,BK0101,BK01,1.0,Brooklyn
1,BK0102,BK01,1.0,Brooklyn
2,BK0103,BK01,1.0,Brooklyn
3,BK0104,BK01,1.0,Brooklyn
4,BK0201,BK02,2.0,Brooklyn
...,...,...,...,...
257,SI0391,SI03,3.0,Staten Island
258,SI9561,SI95,95.0,Staten Island
259,SI9591,SI95,95.0,Staten Island
260,SI9592,SI95,95.0,Staten Island


In [991]:
nta_shelter.head()

,nta_id,cdta_id,cd_number,boro_name,borough,community_districts,total_families_with_children_shelter,high_shelter_flag
0,BK0101,BK01,1.0,Brooklyn,Brooklyn,1.0,0.0,0.0
1,BK0102,BK01,1.0,Brooklyn,Brooklyn,1.0,0.0,0.0
2,BK0103,BK01,1.0,Brooklyn,Brooklyn,1.0,0.0,0.0
3,BK0104,BK01,1.0,Brooklyn,Brooklyn,1.0,0.0,0.0
4,BK0201,BK02,2.0,Brooklyn,Brooklyn,2.0,1708.0,0.0


In [992]:
nta_shelter[['nta_id', 'cdta_id', 'boro_name', 'cd_number', 
                            'total_families_with_children_shelter', 'high_shelter_flag']]

,nta_id,cdta_id,boro_name,cd_number,total_families_with_children_shelter,high_shelter_flag
0,BK0101,BK01,Brooklyn,1.0,0.0,0.0
1,BK0102,BK01,Brooklyn,1.0,0.0,0.0
2,BK0103,BK01,Brooklyn,1.0,0.0,0.0
3,BK0104,BK01,Brooklyn,1.0,0.0,0.0
4,BK0201,BK02,Brooklyn,2.0,1708.0,0.0
...,...,...,...,...,...,...
257,SI0391,SI03,Staten Island,3.0,0.0,0.0
258,SI9561,SI95,Staten Island,95.0,NaN,NaN
259,SI9591,SI95,Staten Island,95.0,NaN,NaN
260,SI9592,SI95,Staten Island,95.0,NaN,NaN


In [993]:
# Join Everything Together at NTA Level
# Start with prioritization as base (197 NTAs)
unified = prio.copy()

# Add EFAP aggregations (left join - some NTAs may have no sites)
unified = unified.merge(nta_efap_agg, left_on='nta_id', right_on='nta_id', how='left')

In [994]:
unified.head()

,nta_id,nta_name,borough,food_insecure_percentage,food_insecure_percentage_rank,unemployment_rate,unemployment_rate_rank,vulnerable_population_percentage,vulnerable_population_percentage_rank,supply_gap,weighted_score,total_sites,pantry_sites,kitchen_sites,weekday_sites,weekend_sites
0,BK0104,East Williamsburg,Brooklyn,35.99,1,6.38,126,12.43,146,2.776626e+06,8.2210,1.0,1.0,0.0,1.0,0.0
1,BX0501,University Heights (South)-Morris Heights,Bronx,29.44,14,11.98,20,19.63,34,1.669389e+06,8.0704,3.0,3.0,0.0,3.0,0.0
2,BX0901,Soundview-Bruckner-Bronx River,Bronx,22.63,36,10.06,32,21.43,25,1.625976e+06,7.6866,4.0,3.0,1.0,1.0,3.0
3,MN1202,Washington Heights (North),Manhattan,24.29,28,12.25,19,18.57,41,1.463457e+06,7.3895,2.0,2.0,1.0,2.0,0.0
4,BK1503,Sheepshead Bay-Manhattan Beach-Gerritsen Beach,Brooklyn,21.11,42,4.91,170,15.95,81,1.907056e+06,7.2775,1.0,1.0,1.0,1.0,0.0


In [995]:
shelter_cd_agg.head()

,borough,community_districts,total_families_with_children_shelter,high_shelter_flag
0,Bronx,1.0,28947,1
1,Bronx,2.0,20047,1
2,Bronx,3.0,30888,1
3,Bronx,4.0,52319,1
4,Bronx,5.0,38815,1


In [996]:
shelter_cd_agg.head(20)

,borough,community_districts,total_families_with_children_shelter,high_shelter_flag
0,Bronx,1.0,28947,1
1,Bronx,2.0,20047,1
2,Bronx,3.0,30888,1
3,Bronx,4.0,52319,1
4,Bronx,5.0,38815,1
5,Bronx,6.0,48601,1
6,Bronx,7.0,11174,0
7,Bronx,8.0,5714,0
8,Bronx,9.0,31536,1
9,Bronx,10.0,13473,0


In [997]:
unified.isna().sum()

nta_id                                    0
nta_name                                  0
borough                                   0
food_insecure_percentage                  0
food_insecure_percentage_rank             0
unemployment_rate                         0
unemployment_rate_rank                    0
vulnerable_population_percentage          0
vulnerable_population_percentage_rank     0
supply_gap                                0
weighted_score                            0
total_sites                              45
pantry_sites                             45
kitchen_sites                            45
weekday_sites                            45
weekend_sites                            45
dtype: int64

### Assumption: Imputing Missing EFAP Site Counts as Zero

After joining the Neighborhood Prioritization dataset (197 NTAs) with the aggregated EFAP dataset (152 NTAs), 45 NTAs contained null values in EFAP-related columns (total_sites, pantry_site_count, kitchen_site_count, weekend_site_count, weekday_site_count). These null values emerged because those NTAs did not have corresponding records in the EFAP dataset.

We assume that these nulls do not represent missing or unreported data, but instead indicate that no EFAP sites are present in those neighborhoods. This interpretation is supported by the join structure: the prioritization dataset defines the universe of neighborhoods, and the EFAP dataset only contains neighborhoods where at least one food site exists. Hence, an unmatched NTA implies zero recorded EFAP supply.

Because of this, we impute these null EFAP values with 0 to accurately reflect the absence of food assistance sites rather than treat them as missing data. Dropping these rows would incorrectly remove valid neighborhoods from the analysis and bias results by excluding areas with potentially the lowest food coverage which are central to our research question.

This assumption allows us to preserve the full neighborhood universe while maintaining analytical consistency in coverage calculations.

In [998]:
unified['total_sites'] = unified['total_sites'].fillna(0).astype(int)
unified['pantry_sites'] = unified['pantry_sites'].fillna(0).astype(int)
unified['kitchen_sites'] = unified['kitchen_sites'].fillna(0).astype(int)
unified['weekday_sites'] = unified['weekday_sites'].fillna(0).astype(int)
unified['weekend_sites'] = unified['weekend_sites'].fillna(0).astype(int)

In [999]:
# Add shelter data (via NTA-CDTA mapping)
unified = unified.merge(
    nta_shelter[['nta_id', 'cdta_id', 'total_families_with_children_shelter', 'high_shelter_flag']],
    on='nta_id',
    how='left'
)

In [1000]:
unified.head()

,nta_id,nta_name,borough,food_insecure_percentage,food_insecure_percentage_rank,unemployment_rate,unemployment_rate_rank,vulnerable_population_percentage,vulnerable_population_percentage_rank,supply_gap,weighted_score,total_sites,pantry_sites,kitchen_sites,weekday_sites,weekend_sites,cdta_id,total_families_with_children_shelter,high_shelter_flag
0,BK0104,East Williamsburg,Brooklyn,35.99,1,6.38,126,12.43,146,2.776626e+06,8.2210,1,1,0,1,0,BK01,0.0,0.0
1,BX0501,University Heights (South)-Morris Heights,Bronx,29.44,14,11.98,20,19.63,34,1.669389e+06,8.0704,3,3,0,3,0,BX05,38815.0,1.0
2,BX0901,Soundview-Bruckner-Bronx River,Bronx,22.63,36,10.06,32,21.43,25,1.625976e+06,7.6866,4,3,1,1,3,BX09,31536.0,1.0
3,MN1202,Washington Heights (North),Manhattan,24.29,28,12.25,19,18.57,41,1.463457e+06,7.3895,2,2,1,2,0,MN12,0.0,0.0
4,BK1503,Sheepshead Bay-Manhattan Beach-Gerritsen Beach,Brooklyn,21.11,42,4.91,170,15.95,81,1.907056e+06,7.2775,1,1,1,1,0,BK15,7847.0,0.0


In [1001]:
unified.shape

(197, 19)

In [1002]:
unified.isnull().sum()

nta_id                                   0
nta_name                                 0
borough                                  0
food_insecure_percentage                 0
food_insecure_percentage_rank            0
unemployment_rate                        0
unemployment_rate_rank                   0
vulnerable_population_percentage         0
vulnerable_population_percentage_rank    0
supply_gap                               0
weighted_score                           0
total_sites                              0
pantry_sites                             0
kitchen_sites                            0
weekday_sites                            0
weekend_sites                            0
cdta_id                                  0
total_families_with_children_shelter     0
high_shelter_flag                        0
dtype: int64

In [1003]:
unified.head()

,nta_id,nta_name,borough,food_insecure_percentage,food_insecure_percentage_rank,unemployment_rate,unemployment_rate_rank,vulnerable_population_percentage,vulnerable_population_percentage_rank,supply_gap,weighted_score,total_sites,pantry_sites,kitchen_sites,weekday_sites,weekend_sites,cdta_id,total_families_with_children_shelter,high_shelter_flag
0,BK0104,East Williamsburg,Brooklyn,35.99,1,6.38,126,12.43,146,2.776626e+06,8.2210,1,1,0,1,0,BK01,0.0,0.0
1,BX0501,University Heights (South)-Morris Heights,Bronx,29.44,14,11.98,20,19.63,34,1.669389e+06,8.0704,3,3,0,3,0,BX05,38815.0,1.0
2,BX0901,Soundview-Bruckner-Bronx River,Bronx,22.63,36,10.06,32,21.43,25,1.625976e+06,7.6866,4,3,1,1,3,BX09,31536.0,1.0
3,MN1202,Washington Heights (North),Manhattan,24.29,28,12.25,19,18.57,41,1.463457e+06,7.3895,2,2,1,2,0,MN12,0.0,0.0
4,BK1503,Sheepshead Bay-Manhattan Beach-Gerritsen Beach,Brooklyn,21.11,42,4.91,170,15.95,81,1.907056e+06,7.2775,1,1,1,1,0,BK15,7847.0,0.0


## 3: FEATURE ENGINEERING - COVERAGE RATIO

COVERAGE RATIO DEFINITION:
We want to measure whether food assistance supply aligns with neighborhood need.

Formula Options:
1. coverage_ratio = total_sites / food_insecure_percentage
   - Interpretation: Sites per unit of food insecurity rate
   - Higher = better coverage relative to need

2. sites_per_1000_insecure = (total_sites / food_insecure_percentage) * 100
   - Scaled version for easier interpretation

3. Binary: has_coverage_gap = 1 if coverage_ratio < median, else 0

We'll use food_insecure_percentage as the denominator because:
- It directly measures food insecurity at the neighborhood level
- It's available for all 197 NTAs
- It's a key component of the prioritization score

In [1004]:
# Create coverage ratio
# Avoid division by zero (though all NTAs have food insecurity > 0)
unified['coverage_ratio'] = unified['total_sites'] / unified['food_insecure_percentage']

In [1005]:
print(unified['coverage_ratio'].describe())


count    197.000000
mean       0.202451
std        0.260725
min        0.000000
25%        0.041135
50%        0.124844
75%        0.295276
max        2.054795
Name: coverage_ratio, dtype: float64


In [1006]:
# Create Priority Groups
# Binary high priority (top 25%)
unified['is_high_priority'] = (
    unified['weighted_score'] >= unified['weighted_score'].quantile(0.75)
).astype(int)

In [1007]:
print(f"\nHigh Priority NTAs (top 25% by weighted_score): {unified['is_high_priority'].sum()}")



High Priority NTAs (top 25% by weighted_score): 50


In [1008]:
# Create Coverage Categories for Logistic Regression

# Binary outcome: Low coverage vs High coverage
# Using median as cutoff
coverage_median = unified['coverage_ratio'].median()
unified['coverage_category'] = (unified['coverage_ratio'] >= coverage_median).astype(int)
unified['coverage_category_label'] = unified['coverage_category'].map({0: 'Low', 1: 'High'})

print(f"Coverage Ratio Median: {coverage_median:.4f}")
print(f"Low Coverage NTAs: {(unified['coverage_category'] == 0).sum()}")
print(f"High Coverage NTAs: {(unified['coverage_category'] == 1).sum()}")

Coverage Ratio Median: 0.1248
Low Coverage NTAs: 98
High Coverage NTAs: 99


- Additional Features

In [1009]:
# Has any kitchen access
unified['has_kitchen'] = (unified['kitchen_sites'] > 0).astype(int)

# Has any weekend access
unified['has_weekend'] = (unified['weekend_sites'] > 0).astype(int)

# Log-transformed features (for modeling, avoiding log(0))
unified['log_total_sites'] = np.log1p(unified['total_sites'])
unified['log_coverage_ratio'] = np.log1p(unified['coverage_ratio'])

In [1010]:
# Normalized priority score (min-max scaling)
unified['priority_normalized'] = (
    (unified['weighted_score'] - unified['weighted_score'].min()) / 
    (unified['weighted_score'].max() - unified['weighted_score'].min())
)

In [1011]:
# create two new features for logistic regression low coverage vs high coverage
unified['is_low_coverage'] = (unified['coverage_category'] == 0).astype(int)
unified['is_high_coverage'] = (unified['coverage_category'] == 1).astype(int)

In [1012]:
unified.columns

Index(['nta_id', 'nta_name', 'borough', 'food_insecure_percentage',
       'food_insecure_percentage_rank', 'unemployment_rate',
       'unemployment_rate_rank', 'vulnerable_population_percentage',
       'vulnerable_population_percentage_rank', 'supply_gap', 'weighted_score',
       'total_sites', 'pantry_sites', 'kitchen_sites', 'weekday_sites',
       'weekend_sites', 'cdta_id', 'total_families_with_children_shelter',
       'high_shelter_flag', 'coverage_ratio', 'is_high_priority',
       'coverage_category', 'coverage_category_label', 'has_kitchen',
       'has_weekend', 'log_total_sites', 'log_coverage_ratio',
       'priority_normalized', 'is_low_coverage', 'is_high_coverage'],
      dtype='object')

In [1013]:
unified.shape

(197, 30)

In [1014]:
unified.head()

,nta_id,nta_name,borough,food_insecure_percentage,food_insecure_percentage_rank,unemployment_rate,unemployment_rate_rank,vulnerable_population_percentage,vulnerable_population_percentage_rank,supply_gap,...,is_high_priority,coverage_category,coverage_category_label,has_kitchen,has_weekend,log_total_sites,log_coverage_ratio,priority_normalized,is_low_coverage,is_high_coverage
0,BK0104,East Williamsburg,Brooklyn,35.99,1,6.38,126,12.43,146,2.776626e+06,...,1,0,Low,0,0,0.693147,0.027406,1.000000,1,0
1,BX0501,University Heights (South)-Morris Heights,Bronx,29.44,14,11.98,20,19.63,34,1.669389e+06,...,1,0,Low,0,0,1.386294,0.097038,0.977019,1,0
2,BX0901,Soundview-Bruckner-Bronx River,Bronx,22.63,36,10.06,32,21.43,25,1.625976e+06,...,1,1,High,1,1,1.609438,0.162762,0.918451,0,1
3,MN1202,Washington Heights (North),Manhattan,24.29,28,12.25,19,18.57,41,1.463457e+06,...,1,0,Low,1,0,1.098612,0.079124,0.873113,1,0
4,BK1503,Sheepshead Bay-Manhattan Beach-Gerritsen Beach,Brooklyn,21.11,42,4.91,170,15.95,81,1.907056e+06,...,1,0,Low,1,0,0.693147,0.046283,0.856022,1,0


### Next Step:
- Statistical Analysis and Modeling

### STATISTICAL TEST SELECTION & ASSUMPTION CHECKING


HYPOTHESIS:
"The ratio of emergency food assistance sites to shelter-related food insecurity 
needs is LOW in HIGH-PRIORITY neighborhoods, indicating a critical service gap."

This hypothesis suggests:
- We are comparing coverage_ratio between two groups: HIGH vs LOW priority NTAs
- We expect coverage_ratio to be LOWER in high-priority neighborhoods
- This is a TWO-GROUP COMPARISON. 

STATISTICAL TEST OPTIONS:
1. Independent Samples t-test (parametric)
   - Compares means between two groups
   - Assumes: Normal distribution, equal variances, independence

2. Mann-Whitney U test (non-parametric)
   - Compares distributions/medians between two groups
   - Does NOT assume normality
   - More robust for skewed data

Based on EDA findings (data violates normality), we should use:
→ Welch's t-test (sensitivity analysis - robust to unequal variances)

In [1015]:
# Split data by priority group
high_priority = unified[unified['is_high_priority'] == 1]['coverage_ratio']
low_priority = unified[unified['is_high_priority'] == 0]['coverage_ratio']

print(f"\nGroup Sizes:")
print(f"  High Priority NTAs: n = {len(high_priority)}")
print(f"  Low Priority NTAs: n = {len(low_priority)}")


Group Sizes:
  High Priority NTAs: n = 50
  Low Priority NTAs: n = 147


#### Assumption 1: Independence
✓ SATISFIED: Each NTA is an independent geographic unit.
  Observations are not paired or repeated measures.
  Each neighborhood appears once in the dataset.

### Assumption 2: Normality (for t-test)




In [1016]:
# Shapiro-Wilk test for normality
stat_high, p_high = shapiro(high_priority)
stat_low, p_low = shapiro(low_priority)

print(f"Shapiro-Wilk Test (High Priority): W = {stat_high:.4f}, p = {p_high:.4f}")
print(f"Shapiro-Wilk Test (Low Priority):  W = {stat_low:.4f}, p = {p_low:.4f}")

if p_high < 0.05 or p_low < 0.05:
    print("→ VIOLATED: At least one group deviates significantly from normality (p < 0.05)")
    print("→ RECOMMENDATION: Use Mann-Whitney U test (non-parametric)")
    normality_satisfied = False
else:
    print("→ SATISFIED: Both groups appear normally distributed")
    normality_satisfied = True

Shapiro-Wilk Test (High Priority): W = 0.8720, p = 0.0001
Shapiro-Wilk Test (Low Priority):  W = 0.7349, p = 0.0000
→ VIOLATED: At least one group deviates significantly from normality (p < 0.05)
→ RECOMMENDATION: Use Mann-Whitney U test (non-parametric)


In [1017]:
# Assumption 3: Homogeneity of Variances (for t-test)
print("\nASSUMPTION 3: HOMOGENEITY OF VARIANCES")
print("-" * 40)

stat_levene, p_levene = levene(high_priority, low_priority)
print(f"Levene's Test: F = {stat_levene:.4f}, p = {p_levene:.4f}")

if p_levene < 0.05:
    print("→ VIOLATED: Variances are significantly different")
    print("→ RECOMMENDATION: Use Welch's t-test (if using parametric) or Mann-Whitney U")
    variance_satisfied = False
else:
    print("→ SATISFIED: Variances are approximately equal")
    variance_satisfied = True

# Print variances
print(f"\nGroup Variances:")
print(f"  High Priority: {high_priority.var():.4f}")
print(f"  Low Priority:  {low_priority.var():.4f}")


ASSUMPTION 3: HOMOGENEITY OF VARIANCES
----------------------------------------
Levene's Test: F = 8.0358, p = 0.0051
→ VIOLATED: Variances are significantly different
→ RECOMMENDATION: Use Welch's t-test (if using parametric) or Mann-Whitney U

Group Variances:
  High Priority: 0.0149
  Low Priority:  0.0841


#### Assumption 4: Scale of Measurement
✓ SATISFIED: Coverage ratio is a continuous variable (ratio scale).
  Appropriate for both t-test and Mann-Whitney U.


### Final Statistical Test Selection
Based on assumption checks: We should use the Welch's t-test
------------------------------------------------------
Reasons:
1. Robust to unequal variances
2. Provides comparison of means (different from Mann-Whitney)
3. Useful for triangulating findings

EFFECT SIZE: 
- For t-test: Cohen's d

In [1018]:
desc_stats = unified.groupby('is_high_priority')['coverage_ratio'].agg([
    'count', 'mean', 'std', 'min', 
    lambda x: x.quantile(0.25),
    'median',
    lambda x: x.quantile(0.75),
    'max'
]).round(4).rename(columns={
    '<lambda_0>': 'Q1',
    '<lambda_1>': 'Q3'
})

In [1019]:
desc_stats

,count,mean,std,min,Q1,median,Q3,max
is_high_priority,,,,,,,,
0,147,0.2256,0.2900,0.0,0.0177,0.1267,0.3193,2.0548
1,50,0.1345,0.1222,0.0,0.0519,0.1009,0.1852,0.6122


## Statistical Analysis

HYPOTHESIS:
-----------
- H0 (Null): Coverage ratios are equal between high and low priority neighborhoods
- H1 (Alternative): Coverage ratios are LOWER in high-priority neighborhoods

TEST: WELCH'S T-TEST

In [1020]:
# Perform Welch's t-test (one-tailed: high priority < low priority)
t_stat, p_value_t_two = ttest_ind(high_priority, low_priority, equal_var=False)
# For one-tailed (high < low), divide by 2 if t is negative
p_value_t_one = p_value_t_two / 2 if t_stat < 0 else 1 - (p_value_t_two / 2)

print(f"Welch's t-test Results:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value (two-tailed): {p_value_t_two:.6f}")
print(f"  p-value (one-tailed, high < low): {p_value_t_one:.6f}")

Welch's t-test Results:
  t-statistic: -3.0863
  p-value (two-tailed): 0.002335
  p-value (one-tailed, high < low): 0.001167


In [1021]:
# Effect size: Cohen's d
pooled_std = np.sqrt(((len(high_priority)-1)*high_priority.std()**2 + 
                       (len(low_priority)-1)*low_priority.std()**2) / 
                      (len(high_priority) + len(low_priority) - 2))
cohens_d = (low_priority.mean() - high_priority.mean()) / pooled_std

print(f"\nEffect Size:")
print(f"  Cohen's d: {cohens_d:.4f}")

# Interpret Cohen's d
if abs(cohens_d) < 0.2:
    d_interpretation = "Small"
elif abs(cohens_d) < 0.5:
    d_interpretation = "Small to Medium"
elif abs(cohens_d) < 0.8:
    d_interpretation = "Medium"
else:
    d_interpretation = "Large"

print(f"  Interpretation: {d_interpretation} effect")


Effect Size:
  Cohen's d: 0.3526
  Interpretation: Small to Medium effect


In [1022]:
# Interpret results
alpha = 0.05
print(f"\nInterpretation (α = {alpha}):")
if p_value_t_one < alpha:
    print(f" REJECT H₀: p = {p_value_t_one:.6f} < {alpha}")
    welch_significant = True
else:
    print(f" FAIL TO REJECT H₀: p = {p_value_t_one:.6f} >= {alpha}")
    welch_significant = False


Interpretation (α = 0.05):
 REJECT H₀: p = 0.001167 < 0.05


- Conclusion: There is a significant difference in covarage ratio between High and Low priority neighborhood 

## Features selections

MODEL PURPOSE:
Logistic Regression to classify neighborhoods as High vs Low coverage ratio.
TARGET VARIABLE: coverage_category (binary: 0 = Low, 1 = High). We need features that represent NEED (demand) and could predict coverage gaps.

In [1023]:
# Features selections for modeling and multicollinearity check
model_features = [
    'food_insecure_percentage',
    'unemployment_rate', 
    'vulnerable_population_percentage',
    'supply_gap',
    'weighted_score',
    'is_high_priority',
    'high_shelter_flag',
    'total_families_with_children_shelter'
]

FEATURE SELECTION RATIONALE:

SELECTED FEATURES (Model 1 - Core):
1. food_insecure_percentage - Direct measure of food need
2. unemployment_rate - Economic vulnerability indicator
3. vulnerable_population_percentage - Demographic vulnerability
4. is_high_priority - Binary priority flag (derived from weighted_score)

SELECTED FEATURES (Model 2 - Extended):
- Will adds:
5. high_shelter_flag - Shelter concentration indicator
6. supply_gap - Existing supply-demand gap metric

Note: weighted_score has high VIF because it's derived from the other variables.

In [1024]:
# Define feature sets
features_core = [
    'food_insecure_percentage',
    'unemployment_rate',
    'vulnerable_population_percentage',
    'is_high_priority'
]

features_extended = [
    'food_insecure_percentage',
    'unemployment_rate',
    'vulnerable_population_percentage',
    'is_high_priority',
    'high_shelter_flag'
]

In [1025]:

def calculate_vif(df, features):
    """Calculate VIF for each feature manually."""
    X = df[features].dropna()
    
    vif_data = pd.DataFrame()
    vif_data['Feature'] = features
    
    vif_values = []
    for i, feature in enumerate(features):
        # Get the feature as target
        y = X[feature].values
        # Get other features as predictors
        other_features = [f for f in features if f != feature]
        X_other = X[other_features].values
        
        # Fit linear regression
        reg = LinearRegression()
        reg.fit(X_other, y)
        r_squared = reg.score(X_other, y)
        
        # VIF = 1 / (1 - R²)
        if r_squared < 1:
            vif = 1 / (1 - r_squared)
        else:
            vif = float('inf')
        vif_values.append(vif)
    
    vif_data['VIF'] = vif_values
    vif_data['Status'] = vif_data['VIF'].apply(
        lambda x: 'OK' if x < 5 else ('High' if x < 10 else 'Severe')
    )
    return vif_data.sort_values('VIF', ascending=False)

In [1026]:
print("\nVIF for Core Features")
vif_core = calculate_vif(unified, features_core)
print(vif_core.to_string(index=False))


VIF for Core Features
                         Feature      VIF Status
        food_insecure_percentage 1.490467     OK
               unemployment_rate 1.417832     OK
                is_high_priority 1.382031     OK
vulnerable_population_percentage 1.054950     OK


In [1027]:
print("\nVIF for Extended Features")
vif_extended = calculate_vif(unified, features_extended)
print(vif_extended.to_string(index=False))


VIF for Extended Features
                         Feature      VIF Status
               unemployment_rate 1.706957     OK
        food_insecure_percentage 1.492743     OK
                is_high_priority 1.409757     OK
               high_shelter_flag 1.230477     OK
vulnerable_population_percentage 1.056455     OK


- All VIF values < 5 for selected features
- No severe multicollinearity detected
    

In [1028]:
shelter_2023_2024.head()

,report_date,borough,community_districts,family_with_children_commercial_hotel,family_with_children_shelter,family_cluster,year,month,Year-Month
759,2024-12-31,Brooklyn,15.0,425,182,0,2024,12,2024-12
760,2024-12-31,Brooklyn,16.0,808,2376,0,2024,12,2024-12
761,2024-12-31,Brooklyn,17.0,319,1600,0,2024,12,2024-12
762,2024-12-31,Brooklyn,18.0,329,819,0,2024,12,2024-12
763,2024-12-31,Manhattan,1.0,527,0,0,2024,12,2024-12


In [1029]:
shelter_2023_2024 = shelter_2023_2024[['report_date', 'year', 'month', 'Year-Month', 'family_with_children_shelter']]

In [1030]:
shelter_2023_2024

,report_date,year,month,Year-Month,family_with_children_shelter
759,2024-12-31,2024,12,2024-12,182
760,2024-12-31,2024,12,2024-12,2376
761,2024-12-31,2024,12,2024-12,1600
762,2024-12-31,2024,12,2024-12,819
763,2024-12-31,2024,12,2024-12,0
...,...,...,...,...,...
2190,2023-01-31,2023,1,2023-01,204
2191,2023-01-31,2023,1,2023-01,88
2192,2023-01-31,2023,1,2023-01,463
2193,2023-01-31,2023,1,2023-01,144


In [1031]:
# export unified dataset for modeling
#unified.to_csv('/Users/Marcy_Student/Desktop/Marcy_Projects/CID_Food_Access/data/clean/unified_dataset_for_modeling.csv', index=False)
# export shelter dataset for time series models
#shelter_2023_2024.to_csv('/Users/Marcy_Student/Desktop/Marcy_Projects/CID_Food_Access/data/clean/shelter_for_TimeSeries.csv')